# DCASE 2020 Task 2 - Unsupervised Anomaly Detection Tutorial

This notebook demonstrates unsupervised anomaly detection on the DCASE 2020 Task 2 dataset.

## What is Unsupervised Anomaly Detection?

Unsupervised anomaly detection learns patterns from **normal data only** and identifies deviations as anomalies. This is ideal for:
- Production scenarios where anomalies are rare
- Cases where labeling anomalies is expensive
- Detecting novel/unknown anomaly types

## Dataset: DCASE 2020 Task 2

**6 machine types**: fan, pump, slider, valve, ToyCar, ToyConveyor
**Training**: ~1,000 normal sounds per machine
**Testing**: ~300-400 mixed (normal + anomaly) sounds per machine

## Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.insert(0, '../src')

from audio_anom.unsupervised_anomaly import (
    LocalOutlierFactorAnomalyDetector,
    IsolationForestAnomalyDetector,
    EllipticEnvelopeAnomalyDetector,
)
from audio_anom.preprocessing_unsupervised import UnsupervisedPreprocessor
from audio_anom.evaluation_unsupervised import ModelComparator
from audio_anom.visualization_unsupervised import (
    plot_confusion_matrix,
    plot_multiple_roc_curves,
    create_results_summary_figure,
)

print("✓ Imports successful")

## 1. Load Data

For this tutorial, we'll use synthetic data. Replace with actual DC2020 data loading.

In [ ]:
def generate_synthetic_data(n_train=1000, n_test_normal=300, n_test_anomaly=200):
    """Generate synthetic audio features."""
    np.random.seed(42)
    
    n_features = 284  # mel_spec (256) + mfcc (26) + stats (2)
    
    # Normal distribution
    mean_normal = np.random.randn(n_features) * 0.5
    X_train_normal = np.random.randn(n_train, n_features) + mean_normal
    X_test_normal = np.random.randn(n_test_normal, n_features) + mean_normal
    
    # Anomaly distribution (shifted)
    mean_anomaly = mean_normal + 2.0
    X_test_anomaly = np.random.randn(n_test_anomaly, n_features) + mean_anomaly
    
    # Combine test data
    X_test = np.vstack([X_test_normal, X_test_anomaly])
    y_test = np.array([0] * n_test_normal + [1] * n_test_anomaly)
    
    # Shuffle
    indices = np.random.permutation(len(y_test))
    X_test = X_test[indices]
    y_test = y_test[indices]
    
    return X_train_normal, X_test, y_test

X_train_normal, X_test, y_test = generate_synthetic_data()

print(f"Training data (normal only): {X_train_normal.shape}")
print(f"Test data (mixed): {X_test.shape}")
print(f"Test labels: {np.sum(y_test == 0)} normal, {np.sum(y_test == 1)} anomaly")

## 2. Preprocess Data

Apply StandardScaler + PCA for dimensionality reduction.

In [ ]:
preprocessor = UnsupervisedPreprocessor(n_components=10, apply_pca=True)
X_train_proc = preprocessor.fit_transform(X_train_normal)
X_test_proc = preprocessor.transform(X_test)

print(f"Original features: {X_train_normal.shape[1]}")
print(f"Reduced features: {X_train_proc.shape[1]}")
print(f"Variance explained: {np.sum(preprocessor.explained_variance_ratio_):.2%}")

## 3. Train Models

We'll train three unsupervised methods:
1. **Local Outlier Factor (LOF)**: Best performer (AUC 0.755+)
2. **Isolation Forest**: Good performer (AUC 0.687+)
3. **Elliptic Envelope**: Moderate performer (AUC 0.643+)

In [ ]:
# Local Outlier Factor
lof_model = LocalOutlierFactorAnomalyDetector(n_neighbors=20, contamination=0.1)
lof_model.fit(X_train_proc)
print("✓ LOF trained")

# Isolation Forest
iforest_model = IsolationForestAnomalyDetector(n_estimators=100, contamination=0.1)
iforest_model.fit(X_train_proc)
print("✓ Isolation Forest trained")

# Elliptic Envelope
envelope_model = EllipticEnvelopeAnomalyDetector(contamination=0.1)
envelope_model.fit(X_train_proc)
print("✓ Elliptic Envelope trained")

models = {
    'LOF': lof_model,
    'Isolation Forest': iforest_model,
    'Elliptic Envelope': envelope_model,
}

## 4. Evaluate Models

In [ ]:
comparator = ModelComparator()

for name, model in models.items():
    comparator.add_model(name, model, X_test_proc, y_test)

# Print comparison
comparison_df = comparator.get_comparison()
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison_df)
print("="*80)

## 5. Visualizations

In [ ]:
# ROC Curves
roc_data = {}
for name, model in models.items():
    y_score = model.anomaly_score(X_test_proc)
    roc_data[name] = (y_test, y_score)

fig = plot_multiple_roc_curves(
    roc_data,
    title="ROC Curves - Model Comparison"
)
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_proc)
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(f"{name}\nConfusion Matrix")
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## 6. Inference Example

In [ ]:
# Use best model (LOF)
best_model = models['LOF']

# Generate new samples
new_normal = np.random.randn(3, X_test.shape[1])
new_anomaly = np.random.randn(3, X_test.shape[1]) * 3

# Preprocess
new_normal_proc = preprocessor.transform(new_normal)
new_anomaly_proc = preprocessor.transform(new_anomaly)

print("\nPredictions on new normal samples:")
for i, sample in enumerate(new_normal_proc, 1):
    pred = best_model.predict(sample.reshape(1, -1))[0]
    score = best_model.anomaly_score(sample.reshape(1, -1))[0]
    print(f"  Sample {i}: {'ANOMALY' if pred == 1 else 'NORMAL'} (score: {score:.4f})")

print("\nPredictions on new anomaly samples:")
for i, sample in enumerate(new_anomaly_proc, 1):
    pred = best_model.predict(sample.reshape(1, -1))[0]
    score = best_model.anomaly_score(sample.reshape(1, -1))[0]
    print(f"  Sample {i}: {'ANOMALY' if pred == 1 else 'NORMAL'} (score: {score:.4f})")

## 7. Save Models

In [ ]:
# Save best model
best_model.save('best_model.pkl')
preprocessor.save('preprocessor.pkl')

print("✓ Models saved!")
print("\nUse deploy_production.py for inference:")
print("  python scripts/deploy_production.py --model best_model.pkl --audio test.wav")

## Summary

### Key Takeaways

✓ **Unsupervised Learning**: Trained on normal data only
✓ **Three Methods**: LOF (best), Isolation Forest, Elliptic Envelope
✓ **Real-World Ready**: Production-ready pipeline with preprocessing
✓ **Strong Performance**: AUC 0.755+ on DCASE 2020

### Next Steps

1. Replace synthetic data with real DCASE 2020 audio
2. Tune hyperparameters (contamination, n_neighbors, n_estimators)
3. Deploy to production using `deploy_production.py`
4. Monitor performance and retrain periodically

### Resources

- [DCASE 2020 Task 2 Dataset](https://zenodo.org/record/3678171)
- [Documentation](../docs/UNSUPERVISED.md)
- [Results Report](../docs/DC2020_RESULTS.md)